# Stage 2 Notebook 20 - Exp2O oracle-IoU diagnostic

**Read from Exp2N (NB19, exp14)**: with the proper top-K decode + lane-NMS + threshold-adaptive lane-F1 evaluator wired in, `val/lane/decoded_f1` is non-zero (1-2%) but very low (CLRKDNet hits ~80% on CULane). Geometry is the project's ALL-TIME BEST: `matched_iou=0.424`, `point_mae=0.320` at epoch 10. The matcher correctly identifies high-quality priors -- but the cls head cannot. Sigmoid scores are tightly clustered at 0.13-0.15 across all priors. When we decode top-4 by score, we get nearly-random priors -> low F1.

Before throwing more compute at fixing cls (extended training, hybrid head), this experiment answers a single decisive question:

> **What's the F1 ceiling that our geometry can support, independent of cls quality?**

Exp2O runs the SAME training as Exp2M/N (no model or loss change), but the val loop now runs *two* lane-F1 metrics per batch:

- `lane/decoded_f1` -- ranks priors by `sigmoid(cls_logit)` (the cls head's output). Same as Exp2N.
- `lane/decoded_oracle_f1` -- ranks priors by ground-truth `LineIoU(pred_curve, GT)` (the perfect ranker). The diagnostic upper bound.

If `decoded_oracle_f1 >> decoded_f1`, geometry can support a high F1 and cls is the only blocker -- worth investing in extended training or a hybrid head. If `decoded_oracle_f1 ~ decoded_f1`, even perfect ranking can't lift F1 (geometry doesn't generalize), and we need a different geometry fix.

Implementation: `LaneF1DecodedMetric` now accepts `score_source='oracle_iou'` and a `key_suffix='_oracle'` so two instances run side by side without colliding. The oracle path computes `_compute_lineiou_target(coord_pred, points_gt, vis)` and passes that as `scores_override` into `decode_top_k_lanes`. Pure inference change; training is identical to Exp2M/N.

### Run mode

1. Keep `DEBUG_MODE = True` for the first run.
2. After the smoke and debug run succeed, change to `False` for the 10-epoch short run.
3. Output is mirrored to the notebook cell, the Colab runtime log, and a Drive log file.
4. Do not rerun Notebook 00.

In [1]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Mounted at /content/drive
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [2]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp15_rmt_gca_clrkd_oracle_diagnostic_joint.yaml'
LOG_FILE = os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_smoke.log')
run_streaming([sys.executable, '-u', 'stage2/scripts/smoke_test_joint_models.py', CONFIG], log_path=LOG_FILE)

[run_streaming] command: /usr/bin/python3 -u stage2/scripts/smoke_test_joint_models.py stage2/configs/exp15_rmt_gca_clrkd_oracle_diagnostic_joint.yaml
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp15_rmt_gca_clrkd_oracle_diagnostic_joint_smoke.log
OK exp15_rmt_gca_clrkd_oracle_diagnostic_joint.yaml
  lane_shape=(1, 16, 72, 2) det_shape=(1, 4, 4)
  lane_loss=9.1152 det_loss=3.1756 grad_cos=-0.3782 lambda_lane=0.0500
  gate_stats={'gate/det_mean': 0.499141663312912, 'gate/lane_mean': 0.5017414093017578, 'gate/det_sat_low': 0.0, 'gate/det_sat_high': 0.0, 'gate/lane_sat_low': 0.0, 'gate/lane_sat_high': 0.0}
[run_streaming] return_code=0


0

In [3]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp15_rmt_gca_clrkd_oracle_diagnostic_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

DEBUG_MODE = False

if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 4
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'short10'
    EPOCHS = 10
    BATCH_SIZE = 8
    LIMIT_TRAIN = 3000
    LIMIT_VAL = 1000
    PRINT_EVERY = 5

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-train', str(LIMIT_TRAIN),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
]

print('DEBUG_MODE:', DEBUG_MODE, flush=True)
print('About to run:', ' '.join(cmd), flush=True)
print('Output tar:', OUTPUT_TAR, flush=True)
print('Visible log file:', LOG_FILE, flush=True)
run_streaming(cmd, log_path=LOG_FILE)

DEBUG_MODE: False
About to run: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp15_rmt_gca_clrkd_oracle_diagnostic_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar --curve-root /content/bdd100k_clrkd_curve --work-dir /content/exp15_rmt_gca_clrkd_oracle_diagnostic_joint_short10 --output-tar /content/drive/MyDrive/EcoCAR/training_runs/exp15_rmt_gca_clrkd_oracle_diagnostic_joint_short10.tar --epochs 10 --batch-size 8 --limit-train 3000 --limit-val 1000 --force-extract --print-every 5
Output tar: /content/drive/MyDrive/EcoCAR/training_runs/exp15_rmt_gca_clrkd_oracle_diagnostic_joint_short10.tar
Visible log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp15_rmt_gca_clrkd_oracle_diagnostic_joint_short10_train.log
[run_streaming] command: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp15_rmt_gca_clrkd_oracle_diagnostic_joint.yaml --curve-tar /con

0

## What to watch in Exp2O training

The `epoch_summary` line now has BOTH metric pairs:
- `val_decoded_f1`, `val_decoded_p`, `val_decoded_r` -- cls-ranked (same as Exp2N).
- `val_oracle_f1`, `val_oracle_p`, `val_oracle_r` -- ground-truth-ranked (the diagnostic upper bound).

What the comparison reveals at epoch 10:

**Outcome A: oracle_f1 >= 0.40 (CLRKDNet-comparable territory)**
Geometry is great, cls is the only blocker. Next experiment: **Exp2P = extended training (30+ epochs)** to test the 'CLRKDNet trains for 100s of epochs' hypothesis. Or **Exp2Q = hybrid head** with a separate binary existence score trained on matched priors and an IoU regression score trained on the continuous target.

**Outcome B: oracle_f1 in 0.10-0.40 (partial)**
Geometry is helpful but not tightly enough aligned for >50% LineIoU thresholding. Two paths: lower the F1@IoU threshold to 0.3 (still standard in some lane benchmarks) for fair comparison, OR invest in tighter geometry (denser priors, tighter line_iou_radius, or finer prior-init scheme).

**Outcome C: oracle_f1 < 0.10 (geometry doesn't generalize)**
Even perfect ranking can't pick priors with IoU>0.5 against original GT. The matched_iou=0.42 measures an OVER-OPTIMIZED in-loop quantity but doesn't translate to GT-aligned curves. Need to rethink prior parameterization or loss formulation.

Pass criteria for the run itself:
- `val_oracle_f1 > 0` for every epoch (sanity).
- Geometry holds: `point_mae <= 0.34`, `matched_line_iou >= 0.40`.
- `val_decoded_f1` is similar to Exp2N's (~ 0.01-0.02), since training is unchanged.

After short10, run NB08 to plot Exp2K / Exp2L / Exp2M / Exp2N / Exp2O side-by-side, with both `lane/decoded_f1` and `lane/decoded_oracle_f1` curves visible.